In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
RAIZ = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, RAIZ)
import numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from src.comun.rutas import DATA_GENERADO, MODELOS
M = json.load(open(os.path.join(MODELOS, "metricas.json"), encoding="utf-8")) if os.path.isfile(os.path.join(MODELOS, "metricas.json")) else {}
for f in os.listdir(MODELOS):
    if f.startswith("metricas_") and f.endswith(".json") and not f.startswith(("metricas_rag", "metricas_agente")):
        M[f[len("metricas_"):-5]] = json.load(open(os.path.join(MODELOS, f), encoding="utf-8"))
def tabla(d, dec=3):
    return pd.DataFrame(d).T.round(dec) if isinstance(d, dict) else pd.DataFrame(d)


# 05. RAG y agente

Recuperación evaluada por separado, filtro de relevancia tipo CRAG, agente único frente a multiagente y juez doble.

In [2]:
R_ = json.load(open(os.path.join(MODELOS, 'metricas_rag.json'), encoding='utf-8')); tabla(R_['recuperacion'])

,recall@4,mrr,documentos_distintos_en_top,n,recall@4_simples,recall@4_compuestas,n_compuestas
denso,0.794,0.725,2.118,34.0,0.750,1.000,6.0
denso + filtro de modulo,0.941,0.831,1.971,34.0,0.929,1.000,6.0
bm25,0.824,0.789,2.382,34.0,0.821,0.833,6.0
bm25 + filtro de modulo,0.882,0.860,1.971,34.0,0.857,1.000,6.0
hibrido,0.882,0.804,2.265,34.0,0.857,1.000,6.0
hibrido + filtro de modulo,0.941,0.892,2.059,34.0,0.929,1.000,6.0
hibrido + multiconsulta,0.853,0.789,2.235,34.0,0.857,0.833,6.0
hibrido + multiconsulta + filtro de modulo,0.941,0.892,2.088,34.0,0.929,1.000,6.0
hibrido + multiconsulta + MMR,0.853,0.787,2.176,34.0,0.857,0.833,6.0
hibrido + multiconsulta + MMR + filtro de modulo,0.941,0.892,2.029,34.0,0.929,1.000,6.0


In [3]:
R_['filtro_relevancia']

{'umbral': 0.7057,
 'exactitud_balanceada': 0.9398395721925134,
 'auc': 0.9786096256684492,
 'similitud_con_respuesta': {'media': 0.7708705882352941, 'min': 0.6925},
 'similitud_fuera_del_indice': {'media': 0.6697090909090909, 'max': 0.7316},
 'n_con_respuesta': 34,
 'n_fuera': 11,
 'con_los_umbrales_actuales': {'umbral_en_uso': 0.7,
  'con_respuesta_aceptadas': 0.9705882352941176,
  'fuera_rechazadas': 0.9090909090909091}}

In [4]:
A = json.load(open(os.path.join(MODELOS, 'metricas_agente_plantillas.json'), encoding='utf-8'))
pd.DataFrame({m: A[m]['resumen'] for m in ('agente_unico', 'multiagente')}).round(3)

,agente_unico,multiagente
n,69.000,69.000
ruteo_correcto,1.000,1.000
respuestas_sin_cifras_inventadas,1.000,1.000
sin_frases_prohibidas,1.000,1.000
cierra_con_tu_decides,1.000,1.000
pedidos_prohibidos_rechazados,1.000,1.000
latencia_media_s,0.933,0.397
latencia_p95_s,3.086,1.118
llamadas_llm_por_respuesta,0.000,0.000
por_plantilla,1.000,1.000


In [5]:
A['agente_unico']['fallos_de_ruteo'], A['multiagente']['fallos_de_ruteo']

([], [])

In [6]:
print(open(os.path.join(RAIZ, 'docs', 'presentacion', 'diagramas', 'grafo_multiagente.md'), encoding='utf-8').read()[:1800])

# Grafo del agente (multiagente)

Generado con `app.get_graph().draw_mermaid()` de LangGraph.

```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	cargar_contexto(cargar_contexto)
	traducir_entrada(traducir_entrada)
	traducir_salida(traducir_salida)
	router(router)
	redactar(redactar)
	juez(juez)
	plantilla(plantilla)
	finalizar(finalizar)
	supervisor(supervisor)
	agente_pronostico(agente_pronostico)
	agente_salud(agente_salud)
	agente_clima(agente_clima)
	agente_tasas(agente_tasas)
	agente_calendario(agente_calendario)
	agente_recomendador(agente_recomendador)
	agente_educador(agente_educador)
	__end__([<p>__end__</p>]):::last
	__start__ --> cargar_contexto;
	agente_calendario --> supervisor;
	agente_clima --> supervisor;
	agente_educador --> supervisor;
	agente_pronostico --> supervisor;
	agente_recomendador --> supervisor;
	agente_salud --> supervisor;
	agente_tasas --> supervisor;
	cargar_contexto --> traducir_entrada;
	finali

## Demo del agente

In [7]:
os.environ['LLM_PROVIDER'] = 'plantillas'
from src.agente_analitico.repositorio import Repositorio
from src.agente_analitico.grafo import construir_grafo, responder
from src.rag.indice import Buscador
sys.path.insert(0, os.path.join(RAIZ, 'eval'))
from evaluar_agente import clientes_de_prueba
repo = Repositorio(); ids = clientes_de_prueba(repo); app = construir_grafo(repo, Buscador(), 'agente_unico')
r = responder(app, ids['independiente_en_zona_del_nino'], '¿Estoy en riesgo por el Fenómeno El Niño?')
print(r['respuesta']); print(r['trazas'], r['cifras_verificadas'], r['rag_estado'])

Hola, Maestro.
En 30 dias esperamos que tu saldo ronde S/ 7,281, con un rango entre S/ 3,912 y S/ 14,733.
Para 2025-11 estimamos ingresos de S/ 4,447 (entre S/ 987 y S/ 9,383) y gastos de S/ 3,450.
Tu salud financiera es 72 de 100 y tu nivel es Maestro.
Lo que mas la baja es tu colchon (meses de ingreso disponibles).
Para subir a campeon te falta mejorar salud financiera: hoy tienes 72.1 y necesitas 75.
Alerta: tu zona (Piura) tiene exposicion alta a lluvias por El Nino segun el ENFEN. Acciones: Revisar el techo y las canaletas de la vivienda y reparar filtraciones antes de las lluvias. Tener un fondo de emergencia para al menos un mes de gastos. Guardar documentos importantes en un lugar alto y en copia digital.
La tasa de referencia del BCRP esta en 4.25% (sin cambio 0.00 puntos frente al mes anterior).
- Vives en Piura, zona con exposicion alta a lluvias por El Nino: prepara tu hogar y un fondo de S/ 7,144. Impacto estimado: S/ 7,144. Por que: el ENFEN mantiene la alerta de El Nino 

## Validación del RAG: chunks, embeddings e índice
Que los vectores se hicieron con el prefijo correcto, están normalizados, alineados con los textos y que no hay duplicados.

In [8]:
RV = json.load(open(os.path.join(MODELOS, 'metricas_rag_validacion.json'), encoding='utf-8'))
print(RV['comprobaciones']); print('embeddings', {k: v for k, v in RV['embeddings'].items() if k in ('modelo', 'dimension_del_indice', 'vectores', 'fragmentos', 'tipo_de_indice')})
print('chunks', RV['chunks']['largo'], 'duplicados', RV['chunks']['duplicados_exactos']); print('autorrecuperación', RV['autorrecuperacion'])

{'dimension_coincide': True, 'vectores_igual_a_fragmentos': True, 'vectores_normalizados': True, 'indice_exacto_de_producto_interno': True, 'indice_alineado_con_los_textos': True, 'consulta_y_documento_se_distinguen': True, 'prefijos_correctos': True, 'sin_fragmentos_diminutos_ni_duplicados': True, 'ningun_fragmento_excede_el_tamano_mas_solape': True, 'metadatos_completos': True, 'solo_licencia_nivel_A': True, 'cada_fragmento_se_recupera_a_si_mismo': True}
embeddings {'modelo': 'nomic-embed-text', 'dimension_del_indice': 768, 'vectores': 3761, 'fragmentos': 3761, 'tipo_de_indice': 'IndexFlatIP'}
chunks {'minimo': 81, 'p5': 264, 'mediana': 775, 'p95': 894, 'maximo': 900} duplicados 0
autorrecuperación {'n': 60, 'el_fragmento_es_el_primero': 0.9833333333333333, 'el_fragmento_esta_en_los_3_primeros': 1.0, 'el_primero_es_del_mismo_documento': 1.0}


## Seguimiento con MLflow
Cada comparación (baseline frente a elegido) y cada modelo de producción quedan como corridas y versiones; el resumen se genera con `eval/registrar_en_mlflow.py`.

In [9]:
print(open(os.path.join(RAIZ, 'docs', 'presentacion', 'mlflow_resumen.md'), encoding='utf-8').read()[:2500])

# Seguimiento de modelos con MLflow

Generado por `eval/registrar_en_mlflow.py` (huella de las metricas `781fc3be87a20fa7`). Experimento **lucas-analitica**; base local `data/generado/mlflow/lucas.db` (no se versiona) o el servidor definido en `MLFLOW_TRACKING_URI`. Abrir la interfaz: `mlflow ui --backend-store-uri sqlite:///data/generado/mlflow/lucas.db`.

## Comparacion por problema: baseline frente al elegido

Cada fila es una corrida de MLflow con la misma metrica y los mismos numeros de `docs/comparativa_modelos.md`; en la interfaz se pueden comparar las corridas de un mismo problema.

| Problema | Metrica | Elegido | Valor del elegido | Baselines y alternativas |
|---|---|---|---|---|
| Segmentación de clientes | Silueta de Gower (mayor es mejor) | **K-Prototypes** | 0.205 | Reglas 2×2: 0.058; K-means (one-hot): 0.204 |
| Pronóstico del saldo a 30 días | Error medio / ingreso mensual (menor es mejor) | **LightGBM + bandas conformales** | 0.426 | Perfil estructural: 0.769; Promedi